In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import argparse
import numpy as np
import random
import scipy
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
import random
from scipy.stats import kendalltau
import pandas as pd

In [2]:
edges=np.genfromtxt("./maayan-vidal/out.maayan-vidal",
                                        dtype=int)
edges=edges-1
adj = sp.coo_matrix((np.ones(edges.shape[0]), (edges[:, 0], edges[:, 1])),
                        shape=(3133, 3133))
adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)

adj_vidal = torch.FloatTensor(np.array(adj.todense()))
list_vidal=list(range(3133))

In [3]:
def min_max_normalization(tensor):
    # 找出数组的最小值和最大值
    min_val = torch.min(tensor)
    max_val = torch.max(tensor)

    # 进行最小 - 最大标准化
    normalized_tensor = (tensor - min_val) / (max_val - min_val)
    return normalized_tensor
# node_feature_BA = min_max_normalization(node_feature_BA)
# node_feature_BA1 = min_max_normalization(node_feature_BA1)

In [7]:
class GNN(torch.nn.Module):
    def __init__(self,input_feature,output_feature):
        super(GNN,self).__init__()
        self.w = nn.Parameter(torch.empty(size=(input_feature,output_feature)))
        self.a= nn.Parameter(torch.empty(size=(1,output_feature)))
        self.sigmod=torch.nn.Sigmoid()
        self.reset_parameters()
    def reset_parameters(self):
        #nn.init.xavier_uniform_(self.w.data,gain=1.414)
        #nn.init.xavier_uniform_(self.a.data,gain=1.414)
        for param in self.parameters():
             nn.init.xavier_uniform_(param)
        
    def forward(self,x,adj):
        adj=torch.Tensor(adj.numpy()+np.identity(adj.shape[0]))
        # adj=torch.FloatTensor(normalize_adj(sp.csr_matrix(adj)+ sp.eye(adj.shape[0])).todense())
        x=torch.mm(adj,x)
        x=torch.mm(x,self.w)
        x=x.add(self.a)
        x=torch.relu(x)
        return x


class CGNN(torch.nn.Module):
    def __init__(self):
        super(CGNN,self).__init__()
        self.layer2=GNN(48,6)
        self.layer3=GNN(6,12)
        # self.fc1=torch.nn.Linear(6,12)
        self.fc1=torch.nn.Linear(12,1)
        # self.fc2=torch.nn.Linear(24,1)
        
        
       
     
    
    
    def forward(self,x,adj,target_nodes):
        #x=self.layer1(x)
        # x=torch.cat([x1,x],dim=-1)
        # x=0.5*x+0.5*x1
        x=self.layer2(x,adj)
        x=self.layer3(x,adj)
       # x=self.layer4(x,adj)
        
        x=x[target_nodes]
        
        x=self.fc1(x.view(x.size(0),-1))
        x=torch.relu(x)
        # x=self.fc2(x)
        # x=torch.relu(x)
        x=x.flatten()
        
        return x

In [32]:
import optuna
import time
import numpy as np
import torch
import torch.optim as optim
from scipy.stats import kendalltau
from sklearn.cluster import KMeans

def objective(trial):
    # 1. 定义超参数搜索空间
    learning_rate = trial.suggest_float(
        "learning_rate", 
        low=0.001, 
        high=0.02, 
        step=0.001  # 学习率：0.001-0.01，步长0.001
    )
    epochs = trial.suggest_int(
        "epochs", 
        low=100, 
        high=2000, # 训练次数：100-1000之间的任意整数
        step=100

    )
    epoch_1=trial.suggest_int(
        "epoch_1", 
        low=100, 
        high=600, # 训练次数：100-1000之间的任意整数
        step=100
    )
    learning_rate1 = trial.suggest_categorical(
        "learning_rate1", 
        [0.01,0.001,0.005]  # 仅允许这三个候选值
    )
    # 2. 初始化模型和优化器
    lable_vidal=np.load("lable/lable_vidal.npy")
    lable_vidal_t=torch.tensor(lable_vidal).float()
    node_feature_vidal=torch.load("EC做损失的特征向量/node_feature_vidal_LSTM_EC_"+str(learning_rate1)+"_"+str(epoch_1)+".pt")
    node_feature_vidal = min_max_normalization(node_feature_vidal)
    # 2. 初始化模型和优化器
    random.seed(17)
    np.random.seed(17)
    torch.manual_seed(17)
    
# 设置聚类数量
    k = 5
    X = node_feature_vidal.detach().numpy()

# 执行 KMeans 聚类
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)

# labels 是每个节点对应的聚类类别，比如 labels[i] 是第i个节点所属的簇编号

# 将节点按簇分类整理
    cluster_nodes = {i: [] for i in range(k)}
    for idx, label in enumerate(labels):
        cluster_nodes[label].append(idx)
#选择训练集的节点
    n_nodes = node_feature_vidal.shape[0]

# 1. 随机采样 5% 的节点索引
    num_train = int(0.05 * n_nodes)+1
    num_pinggu = int(0.05 * n_nodes)+1

    train_idx=[]
    while len(train_idx) < num_train:
        for i in range(k):
            if cluster_nodes[i]:  # 确保当前簇中还有节点可取
                node = random.choice(cluster_nodes[i])  # 随机选一个
                train_idx.append(node)
                cluster_nodes[i].remove(node)
                if len(train_idx) >= num_train:
                    break

    pinggu_idx=[]
    while len(pinggu_idx) < num_train:
        for i in range(k):
            if cluster_nodes[i]:  # 确保当前簇中还有节点可取
                node = random.choice(cluster_nodes[i])  # 随机选一个
                pinggu_idx.append(node)
                cluster_nodes[i].remove(node)
                if len(pinggu_idx) >= num_train:
                    break

# 2. 构造训练集（只选取部分节点特征）
    train_vidal = node_feature_vidal[train_idx]
    train_vidal_lable = lable_vidal_t[train_idx]
# 3. 测试集就是整个 node_feature_BA
    
    pinggu_vidal = node_feature_vidal[pinggu_idx]
    pinggu_vidal_lable = lable_vidal_t[pinggu_idx]



    random.seed(17)
    np.random.seed(17)
    torch.manual_seed(17)
    model = CGNN()  # 实例化你的模型

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=5e-4  # 保持权重衰减不变
    )
    
    # 3. 训练模型
    for epoch in range(epochs):
        # 训练过程（复用你的train函数逻辑）
        model.train()
        output = model(node_feature_vidal.data, adj_vidal, train_idx)
        loss_train = torch.nn.functional.mse_loss(output, train_vidal_lable)
        optimizer.zero_grad()
        loss_train.backward()
        optimizer.step()
        
        # 每100轮打印一次中间结果（可选）
        if (epoch + 1) % 100 == 0:
            print(f"当前参数组合: 学习率={learning_rate}, 训练次数={epochs}, 轮次={epoch+1}, 训练损失={loss_train.item():.4f}")
    
    # 4. 在验证集上评估RMSE
    model.eval()
    with torch.no_grad():  # 关闭梯度计算，节省内存
        pinggu_pre = model(node_feature_vidal.data, adj_vidal, pinggu_idx)
        # 转换为numpy数组计算RMSE
        pinggu_pre_np = pinggu_pre.detach().numpy()
        pinggu_true=pinggu_vidal_lable.detach().numpy()
        # print(kendalltau(pinggu_pre_np,pinggu_pre_np))
        rmse=kendalltau(pinggu_pre_np,pinggu_true).statistic
        # rmse = np.sqrt(np.mean((pinggu_true - pinggu_pre_np) **2))
    
    # 5. 返回需要最小化的RMSE（Optuna会寻找最小RMSE对应的参数）
    return rmse

if __name__ == '__main__':
    # 创建优化实例，方向为"最小化"RMSE
    study = optuna.create_study(direction="maximize")
    # 尝试20组参数组合
    study.optimize(objective, n_trials=80)
    
    # 输出最优结果
    print("\n优化完成！")
    print(f"最佳参数组合: {study.best_params}")
    print(f"最小RMSE: {study.best_value:.4f}")
    print(f"最佳试验编号: {study.best_trial.number}")

[I 2025-09-18 09:54:00,587] A new study created in memory with name: no-name-d30e29d4-dab8-47e9-a839-070c93fdcb7e


当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=100, 训练损失=166.2220
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=200, 训练损失=130.4152
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=300, 训练损失=112.6777
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=400, 训练损失=96.0559
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=500, 训练损失=84.4313
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=600, 训练损失=77.1096
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=700, 训练损失=72.1926
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=800, 训练损失=70.0080
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=900, 训练损失=67.4136
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=1000, 训练损失=65.1158
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=1100, 训练损失=59.9659
当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=1200, 训练损失=57.8333


[I 2025-09-18 09:58:05,781] Trial 0 finished with value: 0.8471752491685516 and parameters: {'learning_rate': 0.010000000000000002, 'epochs': 1300, 'epoch_1': 200, 'learning_rate1': 0.01}. Best is trial 0 with value: 0.8471752491685516.


当前参数组合: 学习率=0.010000000000000002, 训练次数=1300, 轮次=1300, 训练损失=67.9252
当前参数组合: 学习率=0.003, 训练次数=1000, 轮次=100, 训练损失=188.5432
当前参数组合: 学习率=0.003, 训练次数=1000, 轮次=200, 训练损失=156.1244
当前参数组合: 学习率=0.003, 训练次数=1000, 轮次=300, 训练损失=138.7810
当前参数组合: 学习率=0.003, 训练次数=1000, 轮次=400, 训练损失=125.9888
当前参数组合: 学习率=0.003, 训练次数=1000, 轮次=500, 训练损失=116.2355
当前参数组合: 学习率=0.003, 训练次数=1000, 轮次=600, 训练损失=106.9308


[W 2025-09-18 09:59:53,797] Trial 1 failed with parameters: {'learning_rate': 0.003, 'epochs': 1000, 'epoch_1': 500, 'learning_rate1': 0.005} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\Administrator\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_20820\2234478618.py", line 110, in objective
    output = model(node_feature_vidal.data, adj_vidal, train_idx)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\site-packages\torch\nn\modules\module.py", line 1532, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\site-packages\torch\nn\modules\module.py", line 1541, in _call_impl
    return forward_call(*args, *

KeyboardInterrupt: 

In [9]:
node_feature_vidal=torch.load("EC做损失的特征向量/node_feature_vidal_LSTM_EC_0.005_500.pt")
node_feature_vidal = min_max_normalization(node_feature_vidal)

In [11]:
lable_vidal=np.load("lable/lable_vidal.npy")
lable_vidal_t=torch.tensor(lable_vidal).float()

In [13]:
import numpy as np
from sklearn.cluster import KMeans
# 假设 node_feature_jazz 是你的节点特征矩阵，形状是 (num_nodes, feature_dim)
random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
    
# 设置聚类数量
k = 5
X = node_feature_vidal.detach().numpy()

# 执行 KMeans 聚类
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(X)

# labels 是每个节点对应的聚类类别，比如 labels[i] 是第i个节点所属的簇编号

# 将节点按簇分类整理
cluster_nodes = {i: [] for i in range(k)}
for idx, label in enumerate(labels):
    cluster_nodes[label].append(idx)
#选择训练集的节点
n_nodes = node_feature_vidal.shape[0]

# 1. 随机采样 5% 的节点索引
num_train = int(0.05 * n_nodes)+1

train_idx=[]
while len(train_idx) < num_train:
    for i in range(k):
        if cluster_nodes[i]:  # 确保当前簇中还有节点可取
            node = random.choice(cluster_nodes[i])  # 随机选一个
            train_idx.append(node)
            cluster_nodes[i].remove(node)
            if len(train_idx) >= num_train:
                break

pinggu_idx=[]
while len(pinggu_idx) < num_train:
    for i in range(k):
        if cluster_nodes[i]:  # 确保当前簇中还有节点可取
            node = random.choice(cluster_nodes[i])  # 随机选一个
            pinggu_idx.append(node)
            cluster_nodes[i].remove(node)
            if len(pinggu_idx) >= num_train:
                break

# 2. 构造训练集（只选取部分节点特征）
train_vidal = node_feature_vidal[train_idx]
train_vidal_lable = lable_vidal_t[train_idx]
# 3. 测试集就是整个 node_feature_BA
    
pinggu_vidal = node_feature_vidal[pinggu_idx]
pinggu_vidal_lable = lable_vidal_t[pinggu_idx]

In [15]:
import time
import numpy as np
import torch
import torch.optim as optim
from scipy.stats import kendalltau

# 确保随机种子完全一致
random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
# 对于CUDA环境，还需固定cuda随机种子
if torch.cuda.is_available():
    torch.cuda.manual_seed(17)
    torch.cuda.manual_seed_all(17)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train(epoch, model, optimizer, node_feature, adj, train_idx, train_label):
    model.train()
    output = model(node_feature.data, adj, train_idx)
    loss = torch.nn.functional.mse_loss(output, train_label)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

if __name__ == '__main__':
    # 确保数据与Optuna调优时完全一致
    # （需与objective函数中使用的node_feature_vidal、adj_vidal等保持相同）
    # 假设以下变量已正确定义：
    # node_feature_vidal, adj_vidal, train_idx, train_vidal_lable, sum_idx, sum_vidal_lable
    
    # 初始化模型和优化器（与Optuna中完全一致）
    model = CGNN()  # 重新初始化，确保权重初始状态一致
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.01,  # Optuna找到的最佳学习率
        weight_decay=5e-4
    )
    
    # 记录总训练时间
    t_total = time.time()
    loss_values = []
    
    # 训练600轮（与Optuna最佳参数一致）
    for epoch in range(3000):
        loss = train(epoch, model, optimizer, node_feature_vidal, adj_vidal, train_idx, train_vidal_lable)
        loss_values.append(loss)
        
        # 每100轮打印训练损失（与Optuna中保持一致的监控频率）
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}, 训练损失: {loss:.4f}")
    
    # 关键：添加与Optuna中完全一致的评估步骤
    model.eval()
    with torch.no_grad():
        # 使用相同的评估索引和标签
        pinggu_pre = model(node_feature_vidal.data, adj_vidal, pinggu_idx)
        pinggu_pre_np = pinggu_pre.detach().numpy()
        pinggu_true = pinggu_vidal_lable.detach().numpy()
        # 计算Kendall系数（与Optuna中评估指标一致）
        kendall = kendalltau(pinggu_pre_np, pinggu_true).statistic
        print(f"最终Kendall系数: {kendall:.6f}")
    
    print("Optimization Finished!")
    print(f"Total time elapsed: {time.time() - t_total:.2f}s")

Epoch 100, 训练损失: 163.5026
Epoch 200, 训练损失: 117.1304
Epoch 300, 训练损失: 92.9884
Epoch 400, 训练损失: 73.9739
Epoch 500, 训练损失: 67.7280
Epoch 600, 训练损失: 59.9754
Epoch 700, 训练损失: 56.7584
Epoch 800, 训练损失: 52.4124
Epoch 900, 训练损失: 51.4848
Epoch 1000, 训练损失: 49.4110
Epoch 1100, 训练损失: 48.2229
Epoch 1200, 训练损失: 47.2223
Epoch 1300, 训练损失: 46.4631
Epoch 1400, 训练损失: 46.7488
Epoch 1500, 训练损失: 47.8683
Epoch 1600, 训练损失: 44.5308
Epoch 1700, 训练损失: 43.9852
Epoch 1800, 训练损失: 44.1594
Epoch 1900, 训练损失: 43.9345
Epoch 2000, 训练损失: 42.8725
Epoch 2100, 训练损失: 43.4616
Epoch 2200, 训练损失: 42.3353
Epoch 2300, 训练损失: 62.2341
Epoch 2400, 训练损失: 41.8822
Epoch 2500, 训练损失: 42.0095
Epoch 2600, 训练损失: 41.4879
Epoch 2700, 训练损失: 42.0430
Epoch 2800, 训练损失: 41.1293
Epoch 2900, 训练损失: 55.2324
Epoch 3000, 训练损失: 40.8042
最终Kendall系数: 0.867811
Optimization Finished!
Total time elapsed: 841.01s


In [17]:
lable_vidal=np.load("lable/lable_vidal.npy")
lable_vidal_t=torch.tensor(lable_vidal).float()

In [19]:
def MI(res):
    a=pd.DataFrame(res.detach().numpy())
    x=len(a)
    a.rank(axis=0,method='min',numeric_only=None,
    na_option='keep',ascending=True,pct=False)
    b=a.iloc[:,0].value_counts()
    y=0
    for i in range(len(b)):
        y=y+b.iloc[i]*(b.iloc[i]-1)
    ans=(1-y/(x*(x-1)))*(1-y/(x*(x-1)))
    return ans

In [23]:
model_cgnn=model
model_cgnn.eval()
start=time.time()
# output = model_cgnn(node_feature_vidal.data,adj_vidal,node_feature_vidal.data,node_feature_vidal.data)
output = model_cgnn(node_feature_vidal.data,adj_vidal,list_vidal)
# for i in train_idx:
#     output[i]=lable_vidal_t[i]
# for i in pinggu_idx:
#     output[i]=lable_vidal_t[i]

end=time.time()
print(end-start)
output_rank=output.detach().numpy().argsort()

#print(output_rank)
#print(label_rank)
loss_train = torch.nn.functional.mse_loss(output,lable_vidal_t)
print(loss_train)
print(kendalltau(output.detach().numpy(),lable_vidal_t))
print(MI(output))

0.15640807151794434
tensor(28.8761, grad_fn=<MseLossBackward0>)
SignificanceResult(statistic=0.812914324280155, pvalue=0.0)
0.9999735035114483


In [19]:
def calculate_top_k_jaccard(list1, list2,length):
    """
    计算两个列表按影响力排序后不同Top-K集合的Jaccard相似度
    
    参数:
    list1, list2: 节点影响力列表，下标为节点ID，值为影响力
    k_list: 需要计算的K值列表，默认为[100, 200, 300]
    
    返回:
    字典，键为K，值为对应Jaccard相似度
    """
    # 将列表转换为NumPy数组
    k_list = [int(length*0.05),int(length*0.06),int(length*0.07),int(length*0.08),int(length*0.09),int(length*0.10),int(length*0.11),int(length*0.12),int(length*0.13),int(length*0.14),int(length*0.15),int(length*0.16),int(length*0.17),int(length*0.18),int(length*0.19),int(length*0.20),int(length*0.21),int(length*0.22),int(length*0.23),int(length*0.24),int(length*0.25),int(length*0.26),int(length*0.27),int(length*0.28),int(length*0.29),int(length*0.30),int(length*0.31),int(length*0.32),int(length*0.33),int(length*0.34),int(length*0.35),int(length*0.36),int(length*0.37),int(length*0.38),int(length*0.39),int(length*0.40),int(length*0.41),int(length*0.42),int(length*0.43),int(length*0.44),int(length*0.45),int(length*0.46),int(length*0.47),int(length*0.48),int(length*0.49),int(length*0.50)]
    array1 = np.array(list1)
    array2 = np.array(list2)
    
    # 对数组排序并获取节点ID（argsort默认升序，加负号转为降序）
    sorted_indices1 = np.argsort(-array1)  # array1中影响力从大到小的节点ID
    # print(sorted_indices1)
    sorted_indices2 = np.argsort(-array2)  # array2中影响力从大到小的节点ID
    # print(sorted_indices2)
    # 计算各Top-K的Jaccard相似度
    results = {}
    for k in k_list:
        # 取前k个节点ID形成集合
        top_k_set1 = set(sorted_indices1[:k])
        top_k_set2 = set(sorted_indices2[:k])
        
        # 计算交集和并集大小
        intersection = len(top_k_set1 & top_k_set2)
        union = len(top_k_set1 | top_k_set2)
        
        # 计算Jaccard相似度
        jaccard = intersection / union if union > 0 else 0
        results[k] = jaccard
    
    return results

In [20]:
length=len(output)
k_list = [int(length*0.05),int(length*0.1),int(length*0.15),int(length*0.2),int(length*0.25),int(length*0.3),int(length*0.35),int(length*0.4),int(length*0.45),int(length*0.5)]
similarities = calculate_top_k_jaccard(output.detach().numpy(), lable_vidal_t,length)
for k, sim in similarities.items():
    print(sim)

0.8352941176470589
0.8067632850241546
0.7877551020408163
0.7730496453900709
0.7784810126582279
0.7834757834757835
0.8057742782152231
0.7814726840855107
0.8008849557522124
0.7914110429447853
0.7798861480075902
0.776595744680851
0.7703826955074875
0.7648902821316614
0.7525773195876289
0.7535014005602241
0.7496671105193076
0.7576530612244898
0.7647058823529411
0.7754137115839244
0.7795454545454545
0.777292576419214
0.7733473242392445
0.7699293642785066
0.7803921568627451
0.775047258979206
0.7751371115173674
0.7829181494661922
0.7779690189328744
0.7764804003336113
0.7879282218597063
0.786053882725832
0.787201233616037
0.7921686746987951
0.7955882352941176
0.8041756659467243
0.8084507042253521
0.8150448585231194
0.8190411883862255
0.8275862068965517
0.8263123784834737
0.8380102040816326
0.8411507191994997
0.8441717791411043
0.8460613349368611
0.8499704666272888


In [25]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

def calculate_node_ranking(node_scores):
    # 1. 处理纯得分列表（自动生成节点ID，如node_0, node_1...）
    if isinstance(node_scores, list) and all(isinstance(x, (int, float)) for x in node_scores):
        df = pd.DataFrame({
            'node': [f'node_{i}' for i in range(len(node_scores))],
            'score': node_scores
        })

    # 2. 按得分降序排序（影响力越高，得分越高，排名越靠前）
    df_sorted = df.sort_values(by='score', ascending=False).reset_index(drop=True)

    # 3. 计算稠密排名（同得分同排名）
    df_sorted['rank'] = df_sorted['score'].rank(method='dense', ascending=False).astype(int)

    return df_sorted

def calculate_ccdf_full_and_sample(df_ranked, N_total, step=40):

    # ========== 第一步：补全1~N_total所有位次的CCDF值 ==========
    # 1. 统计每个实际排名的节点数和累计数
    rank_count = df_ranked.groupby('rank')['node'].count().reset_index(name='n_i')
    rank_count['sum_n_i'] = rank_count['n_i'].cumsum()
    rank_count['ccdf'] = 1 - rank_count['sum_n_i'] / N_total

    # 2. 构建1~N_total的完整位次映射
    full_rank_list = list(range(1, N_total + 1))  # 1到网络总节点数的所有位次
    full_ccdf_list = []
    current_ccdf = 1.0  # 初始值
    rank_to_ccdf = dict(zip(rank_count['rank'], rank_count['ccdf']))

    # 补全每个位次的CCDF值（缺失位次继承前值）
    for r in full_rank_list:
        if r in rank_to_ccdf:
            current_ccdf = rank_to_ccdf[r]
        full_ccdf_list.append(current_ccdf)

    # 构建完整位次的DataFrame
    full_ccdf_df = pd.DataFrame({
        'rank': full_rank_list,  # 1~N_total的所有位次
        'ccdf': full_ccdf_list
    })

    # ========== 第二步：按步长抽取（基于完整位次） ==========
    total_full_ranks = len(full_ccdf_df)  # 等于N_total
    # 生成基础采样索引（步长为step）
    sample_indices = list(range(0, total_full_ranks, step))
    # 补充最后一个位次的索引
    last_index = total_full_ranks - 1
    if last_index not in sample_indices:
        sample_indices.append(last_index)
    # 去重+排序
    sample_indices = sorted(list(set(sample_indices)))

    # 按索引抽取数据
    sampled_ccdf_df = full_ccdf_df.iloc[sample_indices].reset_index(drop=True)
    sampled_ccdf_values = sampled_ccdf_df['ccdf'].tolist()
    sampled_ranks = sampled_ccdf_df['rank'].tolist()

    return full_ccdf_df, sampled_ccdf_df, sampled_ccdf_values, sampled_ranks

In [27]:
output=output.detach().numpy().tolist()
N_total = len(output)
print(N_total)
    # 步骤1：计算节点排名
df_ranked = calculate_node_ranking(output)
# print(df_ranked)
# total_ranks_actual = len(df_ranked.groupby('rank'))
total_ranks_actual= N_total
print(f"实际总排名数：{total_ranks_actual}")

# 步骤2：按步长40抽取CCDF（自动补充最后一个）
full_ccdf_df, sampled_ccdf_df, sampled_ccdf_values, sampled_ranks = calculate_ccdf_full_and_sample(df_ranked, N_total, step=1)

# 输出结果
# print("\n=== 抽取的位次（1~1200）===")


for rank in sampled_ranks:
    print(rank)

print(f"\n=== 最终抽取的CCDF值（共{len(sampled_ccdf_values)}个）===")
for value in sampled_ccdf_values:
    print(round(value, 4))  # 保留4位小数，更整洁
# print(sampled_ccdf_values)

3133
实际总排名数：3133
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273